# Semantic retrieval benchmark — multilingual-e5-base

Objective comparison of **lexical** vs **semantic** retrieval quality for Mobile_mem0's memory model, using `groonga/multilingual-e5-base-Q4_K_M-GGUF` (Q4_K_M, 768 dimensions, mean pooling, `"query: "`/`"passage: "` prefixes) as the reference embedding model — already verified working on-device; see `download_model.py`'s own doc comment for why this repo, not cstr's original one.

Standalone: no Android, no JNI, no Firestore, and no changes to Mobile_mem0's production code. See `benchmark/README.md` in the repo for the full picture — this notebook just runs the same steps `run_benchmark.py` does, cell by cell, entirely on Colab's free CPU tier.

Run every cell top to bottom. Nothing here needs a Hugging Face token (the model repo is public), and nothing downloaded here is written back into the repo.

## 1. Clone the repo and install dependencies

In [ ]:
import os

if os.path.isdir("Mobile_mem0"):
    # Already cloned in this runtime from an earlier cell run — pull the
    # latest instead of re-cloning, so a fix pushed after this session
    # started (e.g. a corrected model repo) actually takes effect. A plain
    # re-run of this cell alone is NOT enough after a git pull, though:
    # Python caches already-imported modules, so also do Runtime -> Restart
    # session, then Run all again, any time the benchmark's own .py files
    # changed upstream.
    !cd Mobile_mem0 && git pull
else:
    !git clone --depth 1 https://github.com/rmant7/Mobile_mem0.git

%cd Mobile_mem0/benchmark
%pip install -q llama-cpp-python huggingface_hub

## 2. Download the GGUF from Hugging Face
Resolves the actual filename at run time (never hardcoded) — see `download_model.py`. Lands in `huggingface_hub`'s own cache on Colab's ephemeral disk, never committed anywhere.

In [ ]:
from download_model import download, resolve_filename

print("Resolved file:", resolve_filename())
model_path = download()
print("Downloaded to:", model_path)

## 3. Load the dataset

In [ ]:
from run_benchmark import load_jsonl, DEFAULT_DATASET_DIR

memories = load_jsonl(DEFAULT_DATASET_DIR / "memories.jsonl")
queries = load_jsonl(DEFAULT_DATASET_DIR / "queries.jsonl")
print(f"{len(memories)} memories, {len(queries)} queries")

from collections import Counter
print("Queries by category:", Counter(q["category"] for q in queries))

## 4. Load the model and build embeddings
Every memory gets `"passage: "`, every query gets `"query: "` — see `embedder.py`. This is the slow cell (CPU inference on Colab's shared cores); expect a few minutes for ~105 memories + ~100 queries.

In [ ]:
from embedder import E5Embedder

embedder = E5Embedder(model_path)
print("Embedding dimension:", embedder.dimension)
assert embedder.dimension == 768, f"expected 768-dimensional embeddings, got {embedder.dimension}"

memory_vectors = {m["id"]: embedder.embed_passage(m["text"]) for m in memories}
query_vectors = {q["id"]: embedder.embed_query(q["text"]) for q in queries}
print(f"Embedded {len(memory_vectors)} memories and {len(query_vectors)} queries")

## 5. Cosine similarity + semantic retrieval
Ranks every memory against every query by cosine similarity — a direct sort for measurement purposes only, not a candidate ranking formula for production.

In [ ]:
from metrics import QueryResult, cosine_similarity

semantic_results = []
for q in queries:
    qvec = query_vectors[q["id"]]
    scored = [(mid, cosine_similarity(qvec, vec)) for mid, vec in memory_vectors.items()]
    scored.sort(key=lambda pair: (-pair[1], pair[0]))
    semantic_results.append(QueryResult(
        query_id=q["id"], category=q["category"],
        ranked_ids=[mid for mid, _ in scored],
        ground_truth_ids=q["ground_truth_ids"],
        scores_by_id=dict(scored),
    ))
print(f"Ranked {len(semantic_results)} queries semantically")

## 6. Lexical retrieval
Python port of Mobile_mem0's own `MemoryRanking.tokenize()`/`overlap()` — see `lexical.py`.

In [ ]:
import lexical

lexical_results = []
for q in queries:
    hits = lexical.rank(q["text"], memories)
    lexical_results.append(QueryResult(
        query_id=q["id"], category=q["category"],
        ranked_ids=[h.memory_id for h in hits],
        ground_truth_ids=q["ground_truth_ids"],
    ))
print(f"Ranked {len(lexical_results)} queries lexically")

## 7. Overall comparison table

In [ ]:
from metrics import aggregate, aggregate_by_category
from run_benchmark import print_table

lex_overall = aggregate(lexical_results)
sem_overall = aggregate(semantic_results)
print_table("Overall", {"lexical": lex_overall, "semantic (base)": sem_overall})
print(f"\nSemantic (base) avg cosine — positive: {sem_overall.avg_cosine_positive:.3f}, negative: {sem_overall.avg_cosine_negative:.3f}")

## 8. Breakdown by category
`low_overlap` is the category that most directly answers whether semantic retrieval adds anything lexical search can't already do; `identifier` is where lexical is expected to hold its own or win.

In [ ]:
lex_by_cat = aggregate_by_category(lexical_results)
sem_by_cat = aggregate_by_category(semantic_results)
for cat in sorted(lex_by_cat.keys()):
    print_table(f"Category: {cat}", {"lexical": lex_by_cat[cat], "semantic (base)": sem_by_cat[cat]})

## 9. Save results (JSON + CSV)

In [ ]:
from pathlib import Path
from run_benchmark import save_results

save_results(
    Path("results"),
    overall={"lexical": lex_overall.as_dict(), "semantic_base": sem_overall.as_dict()},
    by_category={
        "lexical": {c: m.as_dict() for c, m in lex_by_cat.items()},
        "semantic_base": {c: m.as_dict() for c, m in sem_by_cat.items()},
    },
)

# In Colab: download the two files to your machine, or read them back with
# `from google.colab import files; files.download("results/results.csv")`.